In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# NVIDIA Nemotron Reasoning Challenge - Complete EDA & Strategy Guide

> **What this notebook covers:**
> - Full exploratory data analysis of all 6 puzzle types
> - Rule-based solvers that achieve ~99% accuracy on 3 puzzle types - **no GPU needed**
> - Feature engineering and difficulty analysis
> - Model comparison framework and prompting strategies
> - Professional dashboard with live accuracy tracking
> - Roadmap from 0.50 baseline to 0.76+ top score

---

## The Core Insight

This competition looks like an LLM reasoning challenge. But once you decode the data, **3 of 6 puzzle types are fully deterministic** — solvable with simple Python math in microseconds, with 99%+ accuracy. The other 3 require real reasoning. Understanding this split is the key to a winning strategy.

| Puzzle Type | Count | Approach | Achievable Accuracy |
|---|---|---|---|
| Roman numeral | 1,576 | Hard-coded converter | **100%** |
| Physics gravity | 1,597 | Extract g, apply d=½gt² | **99.4%** |
| Unit conversion | 1,594 | Extract ratio, multiply | **99.4%** |
| Text cipher | 1,576 | Character-level mapping | **38–90%** |
| Bit manipulation | 1,602 | Needs LLM reasoning | 50–80%+ |
| Symbol transform | 1,555 | Needs LLM reasoning | 50–70%+ |

## 1. Setup & Data Loading

In [ ]:
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
from collections import Counter, defaultdict
import warnings
warnings.filterwarnings('ignore')

# Color palette 
COLORS = {
    'bit_manipulation': '#185FA5',
    'physics_gravity':  '#0F6E56',
    'unit_conversion':  '#3B6D11',
    'text_cipher':      '#534AB7',
    'numeral_system':   '#3C3489',
    'symbol_transform': '#993C1D',
}
LABELS = {
    'bit_manipulation': 'Bit manipulation',
    'physics_gravity':  'Physics / gravity',
    'unit_conversion':  'Unit conversion',
    'text_cipher':      'Text cipher',
    'numeral_system':   'Numeral system',
    'symbol_transform': 'Symbol transform',
}

train = pd.read_csv('/kaggle/input/competitions/nvidia-nemotron-model-reasoning-challenge/train.csv')
test  = pd.read_csv('/kaggle/input/competitions/nvidia-nemotron-model-reasoning-challenge/test.csv')

print(f'Train: {train.shape}  |  Test: {test.shape}')
train.head(3)

## 2. Puzzle Type Classification

In [ ]:
def classify_puzzle(prompt: str) -> str:
    """Classify puzzle type from prompt text."""
    p = prompt.lower()
    if 'bit manipulation' in p or '8-bit binary' in p:
        return 'bit_manipulation'
    elif 'numeral system' in p:
        return 'numeral_system'
    elif 'encrypt' in p:
        return 'text_cipher'
    elif 'equation' in p or 'solve for' in p:
        return 'symbol_transform'
    elif 'unit conversion' in p:
        return 'unit_conversion'
    elif 'gravitational' in p or 'gravity' in p:
        return 'physics_gravity'
    return 'other'

train['type'] = train['prompt'].apply(classify_puzzle)
test['type']  = test['prompt'].apply(classify_puzzle)

# Feature engineering
train['prompt_len']   = train['prompt'].str.len()
train['answer_len']   = train['answer'].str.len()
train['n_examples']   = train['prompt'].str.count('->')
train['is_numeric_ans'] = pd.to_numeric(train['answer'], errors='coerce').notna()

print('Puzzle type distribution:')
print(train['type'].value_counts().to_string())
print(f'\nTotal: {len(train)}')

## 3. Dashboard - Overview

In [ ]:
fig = plt.figure(figsize=(18, 14))
fig.patch.set_facecolor('#FAFAFA')
gs = GridSpec(3, 3, figure=fig, hspace=0.45, wspace=0.35)

types_ordered = list(COLORS.keys())
counts = [len(train[train['type']==t]) for t in types_ordered]
clrs   = [COLORS[t] for t in types_ordered]
lbls   = [LABELS[t] for t in types_ordered]

# 1. Donut chart 
ax1 = fig.add_subplot(gs[0, 0])
wedges, texts, autotexts = ax1.pie(
    counts, colors=clrs, autopct='%1.1f%%',
    startangle=90, wedgeprops=dict(width=0.55, edgecolor='white', linewidth=1.5),
    pctdistance=0.78
)
for at in autotexts:
    at.set_fontsize(8); at.set_color('white'); at.set_fontweight('bold')
ax1.set_title('Puzzle type distribution', fontsize=11, fontweight='bold', pad=12)
ax1.text(0, 0, f'{len(train):,}\npuzzles', ha='center', va='center', fontsize=9, fontweight='bold')

# 2. Horizontal bar — counts 
ax2 = fig.add_subplot(gs[0, 1:])
bars = ax2.barh(lbls[::-1], counts[::-1], color=clrs[::-1], height=0.6, edgecolor='white')
for bar, cnt in zip(bars, counts[::-1]):
    ax2.text(bar.get_width() + 10, bar.get_y() + bar.get_height()/2,
             f'{cnt:,}', va='center', fontsize=9)
ax2.set_xlabel('Number of puzzles', fontsize=9)
ax2.set_title('Puzzle counts — perfectly balanced dataset', fontsize=11, fontweight='bold')
ax2.spines[['top','right']].set_visible(False)
ax2.set_xlim(0, 1850)

# 3. Prompt length by type 
ax3 = fig.add_subplot(gs[1, :2])
for i, t in enumerate(types_ordered):
    sub = train[train['type']==t]['prompt_len']
    ax3.boxplot(sub, positions=[i], widths=0.5,
                patch_artist=True,
                boxprops=dict(facecolor=COLORS[t], alpha=0.7),
                medianprops=dict(color='white', linewidth=2),
                whiskerprops=dict(linewidth=1.2),
                flierprops=dict(marker='.', markersize=2, alpha=0.3))
ax3.set_xticks(range(len(types_ordered)))
ax3.set_xticklabels([LABELS[t] for t in types_ordered], rotation=20, ha='right', fontsize=8)
ax3.set_ylabel('Prompt length (chars)', fontsize=9)
ax3.set_title('Prompt length by puzzle type (bit manipulation is longest — most complex)', fontsize=11, fontweight='bold')
ax3.spines[['top','right']].set_visible(False)

# 4. Answer length by type 
ax4 = fig.add_subplot(gs[1, 2])
mean_ans_len = [train[train['type']==t]['answer_len'].mean() for t in types_ordered]
ax4.barh(lbls[::-1], [mean_ans_len[types_ordered.index(t)] for t in types_ordered[::-1]],
         color=clrs[::-1], height=0.6)
ax4.set_xlabel('Mean answer length (chars)', fontsize=9)
ax4.set_title('Avg answer\nlength', fontsize=11, fontweight='bold')
ax4.spines[['top','right']].set_visible(False)

# 5. Examples per puzzle 
ax5 = fig.add_subplot(gs[2, :2])
for i, t in enumerate(types_ordered):
    sub = train[train['type']==t]['n_examples']
    ax5.scatter([i]*len(sub), sub,
                color=COLORS[t], alpha=0.04, s=8)
    ax5.plot([i-0.3, i+0.3], [sub.mean(), sub.mean()],
             color=COLORS[t], linewidth=3)
ax5.set_xticks(range(len(types_ordered)))
ax5.set_xticklabels([LABELS[t] for t in types_ordered], rotation=20, ha='right', fontsize=8)
ax5.set_ylabel('Number of examples (→ arrows)', fontsize=9)
ax5.set_title('In-context examples per puzzle (bit manipulation has most — hardest to crack)', fontsize=11, fontweight='bold')
ax5.spines[['top','right']].set_visible(False)

# 6. Answer type pie
ax6 = fig.add_subplot(gs[2, 2])
numeric_n = train['is_numeric_ans'].sum()
text_n = (~train['is_numeric_ans']).sum()
ax6.pie([numeric_n, text_n], labels=['Numeric\n(float/int)', 'Text\n(string)'],
        colors=['#185FA5', '#534AB7'], autopct='%1.1f%%',
        startangle=90, wedgeprops=dict(edgecolor='white', linewidth=2))
ax6.set_title('Answer\nformat split', fontsize=11, fontweight='bold')

plt.suptitle('NVIDIA Nemotron Reasoning Challenge — EDA Dashboard', 
             fontsize=15, fontweight='bold', y=1.01)
plt.savefig('eda_dashboard.png', dpi=150, bbox_inches='tight', facecolor='#FAFAFA')
plt.show()
print('Dashboard saved.')

## Dashboard Insights

The training set contains **9,500 puzzles** split across exactly 6 puzzle types, each contributing roughly 16–17% of the data. This perfect balance is intentional - the competition tests all types equally, so no single type dominates your score. The test set shown in the data is only 3 samples (a demo), but the real evaluation uses several hundred puzzles. Answers split almost evenly between numeric values (57.7%) and text strings (42.3%).

Three patterns stand out immediately. Bit manipulation prompts are nearly **2.5× longer** than algebra prompts (479 vs 195 chars) because they include 8-11 input/output examples each. Answer lengths are tightly constrained per type - binary answers are always exactly 8 characters, Roman numerals range 1-8 chars, and numeric answers are always 3-6 chars. The dataset has zero class imbalance, which means equal weighting in training is the correct choice.

---

## 4. Puzzle Type Deep-Dive - Examples & Rules

In [ ]:
# Print one clean example per type
for t in types_ordered:
    row = train[train['type']==t].iloc[0]
    print(f"{'='*70}")
    print(f"TYPE: {LABELS[t].upper()}")
    print(f"{'='*70}")
    print(row['prompt'][:600])
    print(f"\n→ ANSWER: {row['answer']}")
    print()

## Puzzle Type Insights

All 6 types can be identified with simple keyword matching on the prompt - no ML needed. The keywords are unambiguous: `bit manipulation`, `numeral system`, `encrypt`, `equation`, `unit conversion`, and `gravitational`. This means you can route every puzzle to the right solving strategy before touching a model.

Each puzzle type hides a different Wonderland secret:

- **Bit manipulation** - a unique bitwise rule per puzzle (XOR combinations, rotations, AND/OR masks). The rule must be reverse-engineered from 8–11 examples. This is the hardest type.
- **Physics gravity** - the gravitational constant `g` has been secretly changed. The formula `d = 0.5 × g × t²` is always given explicitly. Extract `g` from the examples, plug in the query `t`.
- **Unit conversion** - a fixed multiplication ratio per puzzle. Divide any output by its input to find the ratio, then apply to the target.
- **Text cipher** - a character-level substitution cipher, unique per puzzle. Map each cipher letter to its plaintext equivalent using the word pairs provided.
- **Numeral system** - standard Roman numerals. The Wonderland secret is just the Roman numeral system with no modification at all.
- **Symbol transform** - character-level pattern mapping on short symbol strings. Two sub-types exist: pure symbols and numeric expressions with custom operators.

---

## 5. Rule-Based Solvers 

Three puzzle types are **fully deterministic** once you understand the hidden rule. The 'secret' Wonderland rules are:
- **Numeral system** → standard Roman numerals
- **Physics gravity** → d = ½·g·t² with a secret g (extract from examples)
- **Unit conversion** → fixed ratio per puzzle (extract from examples)

In [ ]:
#  SOLVER 1: Roman numerals - 100% accuracy
def solve_roman(prompt: str):
    m = re.search(r'Now, write the number (\d+)', prompt)
    if not m: return None
    n = int(m.group(1))
    val = [1000,900,500,400,100,90,50,40,10,9,5,4,1]
    sym = ['M','CM','D','CD','C','XC','L','XL','X','IX','V','IV','I']
    result = ''
    for v, s in zip(val, sym):
        while n >= v:
            result += s; n -= v
    return result

#  SOLVER 2: Physics gravity — 99.4% accuracy

def solve_physics(prompt: str):
    pairs = re.findall(r't\s*=\s*([\d.]+)s.*?distance\s*=\s*([\d.]+)\s*m', prompt)
    if not pairs: return None
    # d = 0.5 * g * t^2  →  g = 2d/t^2
    gs = [2*float(d)/float(t)**2 for t, d in pairs]
    g  = np.mean(gs)
    # Find query t (it appears after 'Now,')
    m = re.search(r'for t\s*=\s*([\d.]+)s', prompt.split('Now,')[-1])
    if not m: return None
    t = float(m.group(1))
    return round(0.5 * g * t**2, 2)

#  SOLVER 3: Unit conversion — 99.4% accuracy

def solve_unit(prompt: str):
    pairs = re.findall(r'([\d.]+)\s*m\s+becomes\s+([\d.]+)', prompt)
    if not pairs: return None
    ratios = [float(o)/float(i) for i, o in pairs if float(i) != 0]
    ratio  = np.mean(ratios)
    m = re.search(r'(?:convert the following measurement:|measurement:)\s*([\d.]+)', prompt)
    if not m:
        m = re.search(r'([\d.]+)\s*m\s*$', prompt.strip())
    if not m: return None
    return round(float(m.group(1)) * ratio, 2)

#  SOLVER 4: Text cipher — 38% (partial)

def solve_cipher(prompt: str):
    lines = [l.strip() for l in prompt.split('\n') if '->' in l]
    letter_map = {}
    for line in lines:
        parts = line.split('->')
        if len(parts) != 2: continue
        for cw, pw in zip(parts[0].split(), parts[1].split()):
            if len(cw) == len(pw):
                for cc, pc in zip(cw, pw):
                    letter_map[cc] = pc
    m = re.search(r'decrypt the following text: (.+)', prompt)
    if not m: return None
    query = m.group(1).strip()
    decoded = ''
    unknown = 0
    for ch in query:
        if ch == ' ': decoded += ' '
        elif ch in letter_map: decoded += letter_map[ch]
        else: decoded += '?'; unknown += 1
    return decoded.replace('?', '') if unknown <= 1 else None

print('Solvers defined. Running accuracy check...')

## 6. Solver Accuracy Benchmark

In [ ]:
def is_correct(pred, truth: str) -> bool:
    if pred is None: return False
    try:
        return abs(float(str(pred)) - float(str(truth))) / max(abs(float(str(truth))), 1e-9) < 1e-3
    except:
        return str(pred).strip().upper() == str(truth).strip().upper()

SOLVERS = [
    ('Numeral system',  'numeral_system',  solve_roman),
    ('Physics gravity', 'physics_gravity', solve_physics),
    ('Unit conversion', 'unit_conversion', solve_unit),
    ('Text cipher',     'text_cipher',     solve_cipher),
]

results = {}
print(f'{'Solver':<22} {'Correct':>8} {'Total':>8} {'Accuracy':>10}')
print('-' * 52)
for name, type_key, fn in SOLVERS:
    subset  = train[train['type'] == type_key]
    correct = sum(1 for _, r in subset.iterrows() if is_correct(fn(r['prompt']), r['answer']))
    acc     = correct / len(subset) * 100
    results[type_key] = (correct, len(subset), acc)
    print(f'{name:<22} {correct:>8,} {len(subset):>8,} {acc:>9.1f}%')

# Bit & symbol: 0% without model
for t in ['bit_manipulation', 'symbol_transform']:
    n = len(train[train['type']==t])
    results[t] = (0, n, 0.0)

total_correct = sum(v[0] for v in results.values())
total_all     = sum(v[1] for v in results.values())
print('-' * 52)
print(f'{'Rule-based floor':<22} {total_correct:>8,} {total_all:>8,} {total_correct/total_all*100:>9.1f}%')
print(f'\n  → Rule-based solvers cover ~56% of all puzzles at 99%+ accuracy.')
print(f'  → The remaining 44% (bit + symbol) requires model reasoning.')
print(f'  → If model gets 70% on bit+symbol: expected total = {(total_correct + int(0.70*(results["bit_manipulation"][1]+results["symbol_transform"][1])))/total_all*100:.1f}%')

## Solver Insights

Three of the six types are **fully deterministic** - no language model required:

- **Roman numerals** convert at **100% accuracy** with a standard converter.
- **Physics gravity** hits **99.4%** by extracting the secret `g` and applying `d = 0.5 × g × t²`.
- **Unit conversion** hits **99.4%** by extracting the constant ratio from the examples.

These three types alone cover ~50% of all puzzles. The text cipher solver reaches 38% using character-level mapping - it fails only when the query contains letters not seen in the provided examples.

The rule-based floor - solving Roman, physics, and unit perfectly while scoring zero on the remaining three types - gives a **56.3% overall accuracy with zero model training**. This is already above the competition baseline submission of 0.50. If a fine-tuned model achieves 70% on the remaining types, the combined hybrid system projects to **79.6%** - above the current first place score of 0.76.

---

## 7. Feature Engineering

In [ ]:
# 7.1 Numeric features 
train['n_digits_in_prompt'] = train['prompt'].str.count(r'\d')
train['n_arrows']           = train['prompt'].str.count(r'->')
train['n_binary_chars']     = train['prompt'].str.count(r'[01]')
train['has_boxed']          = train['prompt'].str.contains(r'\\boxed', regex=True)
train['answer_is_binary']   = train['answer'].str.match(r'^[01]{8}$')
train['answer_is_roman']    = train['answer'].str.match(r'^[IVXLCDM]+$')

print('Feature summary by type:')
feat_cols = ['prompt_len', 'n_arrows', 'answer_len', 'is_numeric_ans']
display(
    train.groupby('type')[feat_cols]
         .mean()
         .round(2)
         .rename(index=LABELS)
         .rename(columns={
             'prompt_len':     'Avg prompt len',
             'n_arrows':       'Avg examples',
             'answer_len':     'Avg answer len',
             'is_numeric_ans': 'Numeric answer?'
         })
)

In [ ]:
# 7.2 Symbol transform sub-type analysis 
sym = train[train['type']=='symbol_transform'].copy()
sym['subtype'] = np.where(
    sym['prompt'].str.contains(r'\d+[+\-*/\\|{]\d+', regex=True),
    'Numeric expressions', 'Pure symbol strings'
)
print('Symbol transform sub-types:')
print(sym['subtype'].value_counts())

# 7.3 Bit manipulation: number of examples 
bits = train[train['type']=='bit_manipulation']
print(f'\nBit manipulation — example counts:')
print(bits['n_arrows'].value_counts().sort_index())

# 7.4 Physics gravity: distribution of secret g 
def extract_g(prompt):
    pairs = re.findall(r't\s*=\s*([\d.]+)s.*?distance\s*=\s*([\d.]+)\s*m', prompt)
    if not pairs: return None
    return np.mean([2*float(d)/float(t)**2 for t, d in pairs])

phys = train[train['type']=='physics_gravity'].copy()
phys['secret_g'] = phys['prompt'].apply(extract_g)
print(f'\nPhysics — secret g range: {phys["secret_g"].min():.2f} to {phys["secret_g"].max():.2f}')
print(f'Real g = 9.81 — Wonderland uses {phys["secret_g"].nunique()} distinct values')

# 7.5 Unit conversion: distribution of ratios 
def extract_ratio(prompt):
    pairs = re.findall(r'([\d.]+)\s*m\s+becomes\s+([\d.]+)', prompt)
    if not pairs: return None
    return round(np.mean([float(o)/float(i) for i, o in pairs if float(i) != 0]), 4)

unit = train[train['type']=='unit_conversion'].copy()
unit['ratio'] = unit['prompt'].apply(extract_ratio)
print(f'\nUnit conversion — ratio range: {unit["ratio"].min():.4f} to {unit["ratio"].max():.4f}')
print(f'Number of distinct ratios: {unit["ratio"].nunique()}')

## Feature Engineering Insights

Prompt length is the strongest signal for puzzle complexity. Bit manipulation puzzles average 479 characters because they need many examples to convey the hidden rule. Symbol transform puzzles are the shortest (195 chars) but have the least-structured outputs.

The number of in-context examples (`→` arrows) ranges from 0 for physics and unit types (they use a different format) to an average of 9.5 for bit manipulation. Answer length is almost perfectly constant within each type - this is useful for output validation. If your model predicts a 15-character answer for a bit manipulation puzzle, it is wrong before you even check the content.

---

## 8. Model Comparison Framework

How different approaches perform on each puzzle type.

In [ ]:
# Expected accuracy per type under different approaches
# (based on competition results and rule-based analysis)

approaches = {
    'Zero-shot\n(no fine-tune)':      [0.20, 0.40, 0.40, 0.30, 0.20, 0.25],
    'Few-shot\nprompting':            [0.40, 0.65, 0.65, 0.50, 0.35, 0.40],
    'Rule-based\n(no model)':         [0.00, 0.99, 0.99, 0.38, 1.00, 0.00],
    'SFT only\n(competitor ~0.67)':   [0.60, 0.95, 0.95, 0.70, 0.99, 0.55],
    'SFT + Rules\n(hybrid target)':   [0.60, 0.99, 0.99, 0.80, 1.00, 0.55],
    'SFT + RL\n(top ~0.76)':         [0.75, 0.99, 0.99, 0.85, 1.00, 0.70],
}
type_names = [LABELS[t] for t in types_ordered]

fig, ax = plt.subplots(figsize=(16, 6))
fig.patch.set_facecolor('#FAFAFA')
x  = np.arange(len(type_names))
w  = 0.13
ap_colors = ['#B4B2A9', '#888780', '#185FA5', '#0F6E56', '#534AB7', '#993C1D']

for i, (label, accs) in enumerate(approaches.items()):
    offset = (i - len(approaches)/2 + 0.5) * w
    ax.bar(x + offset, accs, w, label=label, color=ap_colors[i], alpha=0.88, edgecolor='white')

ax.set_xticks(x)
ax.set_xticklabels(type_names, fontsize=9)
ax.set_ylabel('Expected accuracy', fontsize=10)
ax.set_ylim(0, 1.15)
ax.set_title('Expected accuracy per puzzle type — approach comparison', fontsize=13, fontweight='bold')
ax.legend(loc='upper left', fontsize=7.5, ncol=3, framealpha=0.9)
ax.axhline(0.76, color='red', linestyle='--', linewidth=1.2, alpha=0.7, label='Top LB = 0.76')
ax.text(5.6, 0.77, 'Top LB = 0.76', fontsize=8, color='red')
ax.spines[['top','right']].set_visible(False)
ax.set_facecolor('#FAFAFA')

plt.tight_layout()
plt.savefig('model_comparison.png', dpi=150, bbox_inches='tight', facecolor='#FAFAFA')
plt.show()

In [ ]:
# Weighted overall score calculator 
type_weights = {t: len(train[train['type']==t])/len(train) for t in types_ordered}

print('Projected overall accuracy per approach:')
print('-' * 55)
for name, accs in approaches.items():
    overall = sum(acc * type_weights[t] for acc, t in zip(accs, types_ordered))
    print(f'{name.replace(chr(10), " "):<30}  →  {overall:.3f}')
print('-' * 55)

## Model Comparison Insights

The performance gap between approaches is not uniform across types. For Roman, physics, and unit conversion - even a zero-shot model does reasonably well because the patterns are simple. The real differentiation happens on **bit manipulation and symbol transform**, where zero-shot performance is weak (~20-25%) and SFT brings it to ~60%.

---

## 9. Prompting Strategy Analysis

In [ ]:
# Demonstrate the three prompt formats that matter 

sample_row = train[train['type']=='bit_manipulation'].iloc[0]
prompt_body = sample_row['prompt']

print('=' * 70)
print('FORMAT 1: Zero-shot (baseline ~0.20 on bit manipulation)')
print('=' * 70)
zeroshot = f"""{prompt_body}

Provide your final answer in \\boxed{{}} format."""
print(zeroshot[:400])

print('\n' + '=' * 70)
print('FORMAT 2: Chain-of-thought (recommended for SFT training)')
print('=' * 70)
cot = f"""{prompt_body}

Think step by step:
1. Identify the puzzle type.
2. Analyze the transformation pattern from the given examples.
3. Apply the pattern to the query input.
4. State your final answer in \\boxed{{}} format."""
print(cot[:500])

print('\n' + '=' * 70)
print('FORMAT 3: Type-aware system prompt (best for SFT)')
print('=' * 70)
system_aware = """You are an expert at solving Alice's Wonderland logic puzzles.
This is a BIT MANIPULATION puzzle. The transformation is an 8-bit binary rule.
Step 1: Examine each input→output pair and hypothesize the bitwise operation.
Step 2: Verify your hypothesis against all examples.
Step 3: Apply the rule to the query.
Always end with: \\boxed{your_8_bit_answer}"""
print(system_aware)

## Prompting Strategy Insights

Format compliance is the single most important factor at `temperature=0.0`. A model that reasons correctly but outputs the answer without `\boxed{}` scores **zero**. Three prompt formats matter in order of strength:

1. **Zero-shot** - weakest, but establishes format compliance baseline.
2. **Chain-of-thought** with type identification - strong baseline, ~10-15 point improvement.
3. **Type-aware system prompts** - tell the model exactly which algorithm to apply before it starts reasoning.

For fine-tuning, training responses should always include the reasoning chain before the boxed answer. This teaches the model the *think first, answer second* pattern that the competition rewards.

---

## 10. Local Evaluation Framework

Critical: always evaluate locally before submitting. This mirrors the competition metric exactly.

In [ ]:
def extract_answer(text: str) -> str:
    """Extract answer from model output — mirrors competition metric logic."""
    # Priority 1: \boxed{} content
    m = re.search(r'\\boxed\{([^}]*)\}', text)
    if m: return m.group(1).strip()
    
    # Priority 2: Last numeric value
    nums = re.findall(r'-?[\d]+\.?[\d]*', text)
    if nums: return nums[-1]
    
    # Priority 3: Last word
    words = text.strip().split()
    return words[-1] if words else ''


def evaluate_local(model_outputs: dict, ground_truth: pd.DataFrame,
                   rel_tol: float = 1e-4) -> dict:
    """
    Evaluate model predictions against ground truth.
    
    Args:
        model_outputs: {id: raw_model_output_string}
        ground_truth:  DataFrame with columns [id, answer, type]
    Returns:
        dict with overall and per-type accuracy
    """
    records = []
    for _, row in ground_truth.iterrows():
        raw = model_outputs.get(row['id'], '')
        pred = extract_answer(raw)
        truth = str(row['answer'])
        
        try:
            correct = abs(float(pred) - float(truth)) / max(abs(float(truth)), 1e-9) < rel_tol
        except:
            correct = pred.strip().upper() == truth.strip().upper()
        
        records.append({'id': row['id'], 'type': row['type'],
                        'pred': pred, 'truth': truth, 'correct': correct})
    
    df = pd.DataFrame(records)
    overall = df['correct'].mean()
    per_type = df.groupby('type')['correct'].mean().to_dict()
    
    print(f'Overall accuracy: {overall:.4f}')
    print('Per-type accuracy:')
    for t, acc in sorted(per_type.items(), key=lambda x: -x[1]):
        print(f'  {LABELS.get(t, t):<22} {acc:.4f}')
    
    return {'overall': overall, 'per_type': per_type, 'details': df}


# Demo: evaluate rule-based solvers 
solver_map = {
    'numeral_system':  solve_roman,
    'physics_gravity': solve_physics,
    'unit_conversion': solve_unit,
    'text_cipher':     solve_cipher,
}

# Build mock model outputs from rule-based solvers
val = train.sample(frac=0.2, random_state=42).copy()
mock_outputs = {}
for _, row in val.iterrows():
    fn = solver_map.get(row['type'])
    if fn:
        pred = fn(row['prompt'])
        mock_outputs[row['id']] = f'The answer is \\boxed{{{pred}}}' if pred else ''
    else:
        mock_outputs[row['id']] = ''  # No solver for bit_manipulation / symbol_transform

print('Rule-based solver evaluation on 20% held-out split:')
print('=' * 55)
_ = evaluate_local(mock_outputs, val[['id','answer','type']])

## Local Evaluation Insights

The evaluation function must mirror the competition metric exactly - prioritize `\boxed{}` content, fall back to the last numeric value, then the last word. Numeric answers use a relative tolerance of `1e-4`, not exact string matching. This matters for physics and unit conversion where floating point rounding can differ.

Always evaluate on a fixed 20% held-out split with `random_state=42` so scores are comparable across runs. Never submit based on public leaderboard movement alone - the demo test set contains only 3 examples, which is pure noise.

---

# 12. Key Takeaways

## What the data tells us

1. **3 of 6 types are deterministic** - Roman numerals, physics gravity, and unit conversion can be solved with simple Python at 99%+ accuracy. No GPU, no model, no fine-tuning.

2. **The competition is really decided on 2 types** - bit manipulation and symbol transform account for 33% of puzzles and require genuine pattern-recognition reasoning. This is where SFT + RL investments pay off.

3. **Text cipher is partially solvable** - character-level substitution mapping from examples works ~38% of the time with the naive approach. Better cipher-solving logic (frequency analysis, partial decoding) can push this higher.

4. **Dataset is perfectly balanced** - ~1,580 puzzles per type. No class imbalance to handle. Equal weighting in training is correct.

5. **Format is everything** - `temperature=0.0` means deterministic decoding. Wrong `\boxed{}` format = 0 score even if reasoning is correct.

*This notebook is part of the NVIDIA Nemotron Reasoning Challenge. All analysis is based on the provided train.csv data.*